# Gyara: Hausa ASR benchmark on FLEURS (free T4)

**Runtime > Change runtime type > T4 GPU**, then **Runtime > Run all**. Each step
saves a zip of the results so far, so a disconnect does not lose finished work.

| Step | What | Writes | Time on a T4 (rough) |
|---|---|---|---|
| 0 | Install, log in, record the GPU | `runs/g0-smoke/env.json`, `nvidia-smi.txt` | 3-5 min |
| 1 | FLEURS ha_ng test (all) + validation (first 100) | `data/fleurs/` (not committed) | 3-8 min |
| 2 | **G0, speech:** one ~30 s clip through `NCAIR1/Hausa-ASR` | `runs/g0-smoke/asr.json`, `.srt` | 1-2 min |
| 3 | Language forcing chosen on **validation**, never test | `runs/dev/`, `runs/g0-smoke/decoding_choice.json` | 3-5 min |
| 4 | **G1:** Hausa-ASR on the full FLEURS test | `runs/hausa-asr-fleurs/` | 5-15 min |
| 5 | `openai/whisper-small` baseline + paired comparison | `runs/whisper-small-fleurs/`, `runs/hausa-asr-fleurs/compare.json` | 5-15 min |
| 6 | **G0, LLM:** one N-ATLaS prompt, 4-bit | `runs/g0-smoke/natlas.json` | 10-20 min (16 GB download) |
| 7 | N-ATLaS suggestion effect on the full test (paired) | `runs/hausa-asr-fleurs-natlas/` | 45-90 min |
| 8 | Leaderboard + final zip | `runs/leaderboard.md`, `gyara-results.zip` | 1 min |

G1 runs before the N-ATLaS steps on purpose: it is the deadline-critical number,
and the 8B download is the slowest, riskiest step.

Every WER/CER is reported raw, `standard` and `lenient` with a 95% cluster-bootstrap
CI (clusters = FLEURS sentence ids). Comparisons are paired bootstraps on the same
utterances; a non-significant difference is reported as "no reliable difference".

*N-ATLAS is an initiative of the Federal Ministry of Communications, Innovation and Digital Economy, and powered by Awarri Technologies.*

## 0. Setup

In [ ]:
# ---- The only settings you may need to change ---------------------------------
GYARA_OWNER = "Baffaabba"          # GitHub user that hosts the repo
GYARA_BRANCH = "feat/foundation"   # change to "main" once merged
GYARA_EXTRAS = "train,llm"
# --------------------------------------------------------------------------------
GYARA_GIT = f"git+https://github.com/{GYARA_OWNER}/gyara@{GYARA_BRANCH}"

import importlib, importlib.util, os, pathlib, subprocess, sys

# Everything (data/, runs/) is written under WORK, so paths match the repo layout.
for base in ("/content", "/kaggle/working"):
    if os.path.isdir(base):
        WORK = pathlib.Path(base) / "gyara-work"
        break
else:
    WORK = pathlib.Path.cwd() / "gyara-work"
WORK.mkdir(parents=True, exist_ok=True)
os.chdir(WORK)

def pip_install(*args):
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args],
                       capture_output=True, text=True)
    print(r.stdout[-1500:], r.stderr[-3000:])
    return r.returncode == 0

INSTALLED_FROM = None
if importlib.util.find_spec("gyara"):
    INSTALLED_FROM = "already installed in this session"
elif pip_install(f"gyara[{GYARA_EXTRAS}] @ {GYARA_GIT}"):
    INSTALLED_FROM = GYARA_GIT
else:
    print("\n>>> GitHub install FAILED (repo not pushed yet, wrong branch, or private). "
          "Run the next cell to install from a zip instead.")
print("working dir:", WORK)
print("gyara installed from:", INSTALLED_FROM)

In [ ]:
# Fallback, ONLY if the cell above failed: install from a zip of the repo.
# On the laptop, in PowerShell, from E:\Repos\gyara:
#   Compress-Archive -Path gyara, pyproject.toml, README.md -DestinationPath gyara-src.zip -Force
# Then run this cell and pick gyara-src.zip. (Kaggle: add the zip as a dataset and set ZIP_PATH.)
ZIP_PATH = None
if INSTALLED_FROM is None:
    import zipfile
    if ZIP_PATH is None:
        from google.colab import files
        ZIP_PATH = next(iter(files.upload()))
    SRC = pathlib.Path("/content/gyara-src" if os.path.isdir("/content") else WORK.parent / "gyara-src")
    zipfile.ZipFile(ZIP_PATH).extractall(SRC)
    roots = sorted(p.parent for p in SRC.rglob("pyproject.toml"))
    assert roots, "no pyproject.toml in the zip: zip the repo root (gyara/, pyproject.toml, README.md)"
    assert pip_install("-e", f"{roots[0]}[{GYARA_EXTRAS}]"), "zip install failed; see pip output"
    sys.path.insert(0, str(roots[0]))  # editable installs are only importable after a restart
    importlib.invalidate_caches()
    INSTALLED_FROM = f"zip upload: {ZIP_PATH}"
print("gyara installed from:", INSTALLED_FROM)
!gyara --version

In [ ]:
# NCAIR1/* models are gated: accept the terms on each model page on huggingface.co
# (NCAIR1/Hausa-ASR and NCAIR1/N-ATLaS) with the account that owns the token, then
# store a read token as a secret named HF_TOKEN
# (Colab: key icon in the left bar, and switch on "Notebook access"; Kaggle: Add-ons > Secrets).
import json, os, platform, zipfile, glob
token = os.environ.get("HF_TOKEN")
if not token:
    try:
        from google.colab import userdata
        token = userdata.get("HF_TOKEN")
    except Exception:
        try:
            from kaggle_secrets import UserSecretsClient
            token = UserSecretsClient().get_secret("HF_TOKEN")
        except Exception:
            token = None
assert token, "No HF_TOKEN secret found"
os.environ["HF_TOKEN"] = token
from huggingface_hub import login
login(token=token)

import torch, transformers
import gyara
assert torch.cuda.is_available(), "No GPU: Runtime > Change runtime type > T4 GPU"
os.makedirs("runs/g0-smoke", exist_ok=True)
try:
    from importlib.metadata import distribution
    direct_url = json.loads(distribution("gyara").read_text("direct_url.json") or "{}")
except Exception:
    direct_url = {}
ENV = {
    "gyara_version": gyara.__version__,
    "installed_from": INSTALLED_FROM,
    "git_commit": direct_url.get("vcs_info", {}).get("commit_id"),
    "gpu": torch.cuda.get_device_name(0),
    "gpu_memory_gb": round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1),
    "torch": torch.__version__, "transformers": transformers.__version__,
    "python": platform.python_version(),
}
json.dump(ENV, open("runs/g0-smoke/env.json", "w"), indent=2)
print(json.dumps(ENV, indent=2))
!nvidia-smi | tee runs/g0-smoke/nvidia-smi.txt

# Small text results only: never audio, never model weights.
KEEP = {".json", ".jsonl", ".md", ".srt", ".vtt", ".txt", ".yaml"}
def save_results(zip_name="gyara-results.zip", download=True):
    """Zip every small result file under runs/ (paths kept, so unzip at the repo root)."""
    files = [p for p in pathlib.Path("runs").rglob("*")
             if p.is_file() and p.suffix in KEEP
             and not any(part.startswith("checkpoint-") for part in p.parts)
             and ("model" not in p.parts or p.name == "README.md")]  # model card only
    with zipfile.ZipFile(zip_name, "w", zipfile.ZIP_DEFLATED) as z:
        for p in files:
            z.write(p, p.as_posix())
    print(f"{zip_name}: {len(files)} files")
    for p in sorted(files):
        print("  ", p.as_posix())
    if download:
        try:
            from google.colab import files as colab_files
            colab_files.download(zip_name)
        except Exception:
            print(f"Not on Colab: download {WORK / zip_name} from the file browser / Output tab.")

## 1. Data: FLEURS ha_ng (CC-BY-4.0)

`test` is the held-out benchmark. The first 100 `validation` utterances are a dev
slice, used only to choose a decoding setting (step 3).

In [ ]:
!gyara fetch-fleurs --split test
!gyara fetch-fleurs --split validation --limit 100
TEST, DEV = "data/fleurs/test.jsonl", "data/fleurs/validation.jsonl"
assert os.path.exists(TEST) and os.path.exists(DEV)
print(open("data/fleurs/test.summary.json").read())

# Loop-detector false-positive check: no human reference should look like a loop.
from gyara import hallucination, manifest
rows = manifest.read(TEST)
fp = [r.id for r in rows if hallucination.check(r.text, r.duration).flagged]
json.dump({"references": len(rows), "flagged_as_loop": len(fp), "ids": fp},
          open("runs/g0-smoke/loop_check_references.json", "w"), indent=2)
print(f"loop detector flags {len(fp)} of {len(rows)} FLEURS test references (should be 0)")

## 2. G0, speech model: one ~30 s clip through NCAIR1/Hausa-ASR

The test clip closest to 30 s (Whisper's window) goes through the full
transcription path (VAD, decode, loop check) and becomes subtitles.

In [ ]:
import time
from gyara.asr import Transcriber
from gyara.subtitles import build_cues, to_srt

clip = max((r for r in rows if (r.duration or 0) <= 30.0), key=lambda r: r.duration)
t0 = time.perf_counter()
tr = Transcriber("NCAIR1/Hausa-ASR", language="hausa")
segs = tr.transcribe(clip.audio)
secs = time.perf_counter() - t0
G0_TEXT = " ".join(s.text for s in segs if s.text).strip()
g0 = {"model": "NCAIR1/Hausa-ASR", "decoding": tr.info(), "clip_id": clip.id,
      "clip_seconds": clip.duration, "wall_seconds_incl_load": round(secs, 1),
      "reference": clip.text, "hypothesis": G0_TEXT,
      "segments": [s.to_dict() for s in segs], "env": ENV}
json.dump(g0, open("runs/g0-smoke/asr.json", "w", encoding="utf-8"), indent=2, ensure_ascii=False)
open("runs/g0-smoke/asr.srt", "w", encoding="utf-8").write(to_srt(build_cues([s.to_dict() for s in segs])))
print("language forced:", tr.info()["language_used"], "(None = the model could not force Hausa; see the log)")
print("REF:", clip.text)
print("HYP:", G0_TEXT)
print(f"{secs:.1f} s incl. model load, flags: {[s.flags for s in segs]}")
del tr; import gc; gc.collect(); torch.cuda.empty_cache()

## 3. Language forcing, chosen on the dev slice

`NCAIR1/Hausa-ASR`'s model card calls `generate()` with no language token, so it
may have been trained without one. Forcing `<|ha|>` can help or hurt a lot. We try
both settings on 100 **validation** utterances and use the better one on test.
That is 2 decoding configurations tried, recorded below. The test set is never
used to choose.

In [ ]:
!gyara eval --model NCAIR1/Hausa-ASR --manifest {DEV} --out-dir runs/dev --name hausa-asr-dev100-lang-hausa --language hausa
!gyara eval --model NCAIR1/Hausa-ASR --manifest {DEV} --out-dir runs/dev --name hausa-asr-dev100-lang-auto --language auto

def std_wer(run):
    return json.load(open(f"{run}/metrics.json", encoding="utf-8"))["modes"]["standard"]["wer"]["estimate"]

w_hausa = std_wer("runs/dev/hausa-asr-dev100-lang-hausa")
w_auto = std_wer("runs/dev/hausa-asr-dev100-lang-auto")
LANG = "auto" if w_auto < w_hausa else "hausa"   # ties keep the documented default
choice = {"chosen_language": LANG, "configs_tried": 2,
          "dev_set": "FLEURS ha_ng validation, first 100 utterances",
          "dev_standard_wer": {"hausa": w_hausa, "auto": w_auto},
          "rule": "lower standard WER on dev wins; ties keep 'hausa'. Test never used."}
json.dump(choice, open("runs/g0-smoke/decoding_choice.json", "w"), indent=2)
print(json.dumps(choice, indent=2))
save_results("gyara-results-g0.zip")

## 4. G1: Hausa-ASR baseline on the full FLEURS test

This is `runs/hausa-asr-fleurs/metrics.json`, the file G1 is waiting for. A zip
downloads as soon as it finishes.

In [ ]:
!gyara eval --model NCAIR1/Hausa-ASR --manifest {TEST} --name hausa-asr-fleurs --language {LANG}
save_results("gyara-results-g1.zip")

In [ ]:
from IPython.display import Markdown, display
display(Markdown(open("runs/hausa-asr-fleurs/report.md", encoding="utf-8").read()))

## 5. Baseline: openai/whisper-small, and the paired comparison

whisper-small is the model Hausa-ASR was fine-tuned from, decoded with `<|ha|>`
forced (it is multilingual and would otherwise guess the language). The
comparison is a paired bootstrap on the same utterances; `compare.json` and
`compare.md` land in `runs/hausa-asr-fleurs/`.

In [ ]:
!gyara eval --model openai/whisper-small --manifest {TEST} --name whisper-small-fleurs --language hausa
!gyara compare runs/hausa-asr-fleurs runs/whisper-small-fleurs > /dev/null
display(Markdown(open("runs/hausa-asr-fleurs/compare.md", encoding="utf-8").read()))
save_results("gyara-results-g1-compare.zip")

## 6. G0, LLM: one N-ATLaS prompt (4-bit on the T4)

One proofreading request with the G0 transcript from step 2. The raw model output
and the guard's decision are both saved. Downloads ~16 GB on first use.

In [ ]:
from gyara.suggest import Suggester, build_messages, guard

t0 = time.perf_counter()
sug = Suggester("transformers")          # 4-bit NF4 via bitsandbytes on CUDA
sug._load_transformers()
load_s = time.perf_counter() - t0
t1 = time.perf_counter()
raw = sug._chat(build_messages(G0_TEXT))
gen_s = time.perf_counter() - t1
res = guard(G0_TEXT, raw, sug.model_id)
qc = getattr(sug._model.config, "quantization_config", None)
natlas = {
    "model": sug.model_id,
    "quantization": qc.to_dict() if hasattr(qc, "to_dict") else qc,
    "load_seconds": round(load_s, 1), "generate_seconds": round(gen_s, 1),
    "gpu_peak_memory_gb": round(torch.cuda.max_memory_allocated() / 1e9, 2),
    "input": G0_TEXT, "raw_output": raw, "suggestion": res.to_dict(), "env": ENV,
}
json.dump(natlas, open("runs/g0-smoke/natlas.json", "w", encoding="utf-8"), indent=2, ensure_ascii=False, default=str)
print("quantization:", "4-bit" if qc else "NONE (fp16): bitsandbytes missing?")
print("IN :", G0_TEXT)
print("RAW:", raw)
print("guard:", res.accepted_by_guard, "-", res.reason)
del sug; gc.collect(); torch.cuda.empty_cache()
save_results("gyara-results-g0-llm.zip")

## 7. Does N-ATLaS help? Suggestion effect on the full FLEURS test

Re-runs Hausa-ASR (same decoding) and passes every transcript through N-ATLaS
behind the guard. The report's "Suggestion effect" section is a **paired
bootstrap of `hyp` vs `hyp_suggested`** on the same utterances: the upper bound
where a human accepts every suggestion. It is reported whatever it shows,
including "no reliable difference" or a WER increase.

In [ ]:
!gyara eval --model NCAIR1/Hausa-ASR --manifest {TEST} --name hausa-asr-fleurs-natlas --language {LANG} --suggest transformers
save_results("gyara-results-natlas.zip")

# Sanity: the ASR text should match the baseline run (same model, same greedy decode).
def preds(run):
    return {json.loads(l)["id"]: json.loads(l) for l in open(f"{run}/predictions.jsonl", encoding="utf-8")}
a, b = preds("runs/hausa-asr-fleurs"), preds("runs/hausa-asr-fleurs-natlas")
same = sum(a[i]["hyp"] == b[i]["hyp"] for i in a if i in b)
print(f"ASR text identical to the baseline run on {same} of {len(a)} utterances")
m = json.load(open("runs/hausa-asr-fleurs-natlas/metrics.json", encoding="utf-8"))
print("suggest errors:", m["meta"].get("suggest_errors"))
rep = open("runs/hausa-asr-fleurs-natlas/report.md", encoding="utf-8").read()
display(Markdown(rep[rep.find("## Suggestion effect"):]))

## 8. Leaderboard and the final zip

In [ ]:
!gyara leaderboard runs/hausa-asr-fleurs runs/whisper-small-fleurs --out runs/leaderboard.md
save_results("gyara-results.zip")

## What to bring back to the repo

Unzip **`gyara-results.zip` at the repo root** (`E:\Repos\gyara`). It recreates:

- `runs/hausa-asr-fleurs/`: `metrics.json`, `report.md`, `meta.json`,
  `compare.json`, `compare.md` (vs whisper-small), `predictions.jsonl`
- `runs/whisper-small-fleurs/`: `metrics.json`, `report.md`, `meta.json`, `predictions.jsonl`
- `runs/hausa-asr-fleurs-natlas/`: same files; the N-ATLaS effect is in its `report.md` / `metrics.json`
- `runs/g0-smoke/`: `env.json`, `nvidia-smi.txt`, `asr.json`, `asr.srt`, `natlas.json`,
  `decoding_choice.json`, `loop_check_references.json` (the G0 evidence)
- `runs/dev/`: the two 100-utterance validation runs behind the language choice
- `runs/leaderboard.md`

`predictions.jsonl` is git-ignored but keep it locally: `gyara rescore` needs it.
If the session died part-way, the earlier zips (`gyara-results-g0.zip`,
`-g1.zip`, `-g1-compare.zip`, `-g0-llm.zip`, `-natlas.zip`) hold everything
finished up to that point. Unzip the latest one you have.